## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import gensim
import re
import string
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from myNN import NN

## General functions

In [2]:
language_key = os.getenv("LANGUAGE_KEY")
language_endpoint = os.getenv("LANGUAGE_ENDPOINT")
def authenticate_client():
    ta_credential = AzureKeyCredential(language_key)
    text_analytics_client = TextAnalyticsClient(
            endpoint=language_endpoint,
            credential=ta_credential)
    return text_analytics_client

def load_data(file_path):
    df = pd.read_csv(file_path)
    inputs = df["Text"].values
    outputs = df["Sentiment"].values
    return inputs, outputs

def normalize_text(text):

    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = " ".join(text.split())
    return text

def split_data(inputs, outputs, seedNo = 5, trainSize = 0.8):

    np.random.seed(seedNo)
    noSamples = len(inputs)
    indexes = [i for i in range(noSamples)]

    trainSample = np.random.choice(indexes, size = int(trainSize * noSamples), replace=False)
    testSample = [i for i in indexes if i not in trainSample]

    trainInputs = [inputs[i] for i in trainSample]
    trainOutputs = [outputs[i] for i in trainSample]
    testInputs = [inputs[i] for i in testSample]
    testOutputs = [outputs[i] for i in testSample]

    return trainInputs, trainOutputs, testInputs, testOutputs

def accuracy_score_azure(client, test_in, test_out):

    batch_size = 10
    y_pred = []
    y_true = []

    for i in range(0, len(test_in), batch_size):

        batch_texts = test_in[i:i+batch_size]
        batch_true = test_out[i:i+batch_size]

        result = client.analyze_sentiment(batch_texts, show_opinion_mining=True)

        for doc, true_label in zip(result, batch_true):
            if doc.is_error:
                continue

            y_pred.append(doc.sentiment.lower().strip())
            y_true.append(true_label.lower().strip())

    correct = sum(p == t for p, t in zip(y_pred, y_true))

    return correct / len(y_true)


## General Constants

In [3]:
text = [
    "By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement..",
]
text = [normalize_text(t) for t in text]
mixed_reviews_file = os.path.join(os.getcwd(), 'data', 'mixed_reviews.csv')
imdb_dataset_file = os.path.join(os.getcwd(), 'data', 'IMDB Dataset.csv')
client = authenticate_client()
inputs, outputs = load_data(mixed_reviews_file)
train_in_raw, train_out, test_in_raw, test_out = split_data(inputs, outputs)
train_in = [normalize_text(text) for text in train_in_raw]
test_in = [normalize_text(text) for text in test_in_raw]
modelPath = os.path.join(os.getcwd(), 'models', 'GoogleNews-vectors-negative300.bin')
word2VecModel300 = gensim.models.KeyedVectors.load_word2vec_format(modelPath, binary=True)



## Azure Client Sentiment - 50p

In [4]:
def sentiment_analysis_azure_client(client, documents):

    result = client.analyze_sentiment(documents, show_opinion_mining=True)
    docs = [doc for doc in result if not doc.is_error]

    for i, doc in enumerate(docs):
        print(f"Text: {documents[i]}")
        print(f"Sentiment: {doc.sentiment}")
        print(f"Confidence scores: {doc.confidence_scores}")
        print()

sentiment_analysis_azure_client(client, documents = text)

Text: by choosing a bike over a car i’m reducing my environmental footprint cycling promotes ecofriendly transportation and i’m proud to be part of that movement
Sentiment: positive
Confidence scores: {'positive': 0.94, 'neutral': 0.05, 'negative': 0.0}



## Extract characteristics - Bag of Words, TF-IDF, Word2Vec - 50p + other characteristics - 100p

In [5]:
def extract_characteristics_bag_of_words(train_inputs, test_inputs):
    vectorizer = CountVectorizer()
    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)
    print("vocab size: ", len(vectorizer.vocabulary_),  " words")
    print("traindata size: ", len(train_inputs), " emails")
    print("train_features shape: ", train_features.shape)
    print('some words of the vocab: ', vectorizer.get_feature_names_out()[:20])
    print('some features: ', train_features.toarray()[:3])
    print("=======================================================")
    print("testdata size: ", len(test_inputs), " emails")
    print("test_features shape: ", test_features.shape)
    print('some features: ', test_features.toarray()[:3])
    print("=======================================================")
    return train_features, test_features

def extract_characteristics_tfidf(train_inputs, test_inputs):

    vectorizer = TfidfVectorizer(max_features=2000)
    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)

    print('vocab train tfidf: ', vectorizer.get_feature_names_out()[:10])
    print('features train tfidf: ', train_features.toarray()[:3])

    return train_features, test_features, vectorizer

def extract_characteristics_word2vec(model, data):

    v_size = model.vector_size
    features = np.zeros((len(data), v_size))

    for i, phrase in enumerate(data):

        words = [word for word in phrase.split() if len(word) > 2 and word in model.index_to_key]

        if words:

            word_vectors = model[words]
            features[i] = np.mean(word_vectors, axis=0)


    print("features shape(w2v): ", features.shape)
    return features

def extract_other_features(data):
    features = []
    for text in data:
        words = text.split()
        num_words = len(words)
        num_chars = len(text)
        avg_word_len = num_chars / num_words if num_words > 0 else 0
        num_exclam = text.count("!")
        num_question = text.count("?")
        max_word_len = max((len(w) for w in words), default=0)
        num_caps = sum(1 for w in words if w.isupper())
        features.append([
            num_words,
            num_chars,
            num_caps,
            avg_word_len,
            max_word_len,
            num_exclam,
            num_question
        ])
    print("==================OTHER FEATURES================:")
    print(features[:2])
    return features



In [6]:
#BoW
train_features, test_features = extract_characteristics_bag_of_words(train_in, test_in)

#TF-IDF
train_features_tfidf, test_features_tfidf, vectorizer_tfidf = extract_characteristics_tfidf(train_in, test_in)
text_features_tfidf = vectorizer_tfidf.transform(text)

#W2V
train_features_w2v = extract_characteristics_word2vec(word2VecModel300, train_in)
test_features_w2v = extract_characteristics_word2vec(word2VecModel300, test_in)

#Other features
train_other_features = extract_other_features(train_in_raw)
test_other_features = extract_other_features(test_in_raw)

#Data for ANN
# X_train = train_features_tfidf.toarray()
# X_test = test_features_tfidf.toarray()
# X_text = text_features_tfidf.toarray()

# Data 2 for ANN
X_train = np.array(train_features_w2v)
X_test = np.array(test_features_w2v)
X_text = np.array(extract_characteristics_word2vec(word2VecModel300, text))

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_text = scaler.transform(X_text)



vocab size:  595  words
traindata size:  165  emails
train_features shape:  (165, 595)
some words of the vocab:  ['about' 'above' 'abundant' 'ac' 'access' 'across' 'actually' 'added'
 'adjust' 'advertised' 'advised' 'after' 'agreed' 'ahead' 'air' 'aircon'
 'all' 'allow' 'also' 'although']
some features:  [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
testdata size:  42  emails
test_features shape:  (42, 595)
some features:  [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
vocab train tfidf:  ['about' 'above' 'abundant' 'ac' 'access' 'across' 'actually' 'added'
 'adjust' 'advertised']
features train tfidf:  [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
features shape(w2v):  (165, 300)
features shape(w2v):  (42, 300)
==================OTHER FEATURES================:
[[4, 20, 0, 5.0, 6, 0, 0], [11, 53, 0, 4.818181818181818, 8, 0, 0]]
==================OTHER FEATURES================:
[[9, 54, 0, 6.0, 11, 0, 0], [3, 26, 0, 8.666666666666666,

## ANN cu tool - 100p

In [7]:
mlp = MLPClassifier(
    hidden_layer_sizes=(10,),
    activation='relu',
    max_iter=500,
    random_state=45
)

mlp.fit(X_train, train_out)
predictions = mlp.predict(X_test)
accuracy_sklearn = accuracy_score(test_out, predictions)
prediction_cerinta = mlp.predict(X_text)
print(f"Accuracy sklearn MLP: {accuracy_sklearn:.2f}")
print("Predictie la textul din cerinta: ", prediction_cerinta)

Accuracy sklearn MLP: 0.81
Predictie la textul din cerinta:  ['positive']


## ANN cu cod propriu - 300p

In [8]:
le = LabelEncoder()
y_train = le.fit_transform(train_out)
y_test = le.transform(test_out)

nn = NN(
    n_features=X_train.shape[1],
    n_classes=len(np.unique(y_train)),
    n_hidden=10
)
nn.fit(X_train, y_train, max_iters=800, eta=0.15)
pred_test = nn.predict(X_test)
pred_text = nn.predict(X_text)
accuracy_nn = np.mean(pred_test == y_test)
print(f"Accuracy NN manual: {accuracy_nn:.2f}")
print("Predictie la textul din cerinta (NN manual): ", le.inverse_transform(pred_text))

Iteratia 0: loss 0.6925
Iteratia 100: loss 0.1580
Iteratia 200: loss 0.0637
Iteratia 300: loss 0.0472
Iteratia 400: loss 0.0419
Iteratia 500: loss 0.0396
Iteratia 600: loss 0.0384
Iteratia 700: loss 0.0377
Accuracy NN manual: 0.79
Predictie la textul din cerinta (NN manual):  ['positive']


## Concluzie finala
 - NN este mai bun decat Azure Client deoarece este antrenat pe dataset specific iar azure este mai general

In [9]:
print("Accuracy Azure Client: ", accuracy_score_azure(client, test_in, test_out))
print(f"Accuracy sklearn MLP: {accuracy_sklearn:.2f}")
print(f"Accuracy NN manual: {accuracy_nn:.2f}")

Accuracy Azure Client:  0.7380952380952381
Accuracy sklearn MLP: 0.81
Accuracy NN manual: 0.79
